## Loading Packages

In [1]:
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import folium

from tqdm import tqdm
from pathlib import Path

## Demo Trip DF


In [2]:
trip_demo = pd.DataFrame({
    "ride_id": list(range(1, 21)),

    "start_station": [
        "Station A", "Station A", "Station A", "Station A", "Station A",
        "Station B", "Station B", "Station B", "Station B",
        "Station C", "Station C", "Station C",
        "Station D", "Station D", "Station D",
        "Station E", "Station E",
        "Station A", "Station B", "Station C"
    ],

    "end_station": [
        "Station B", "Station B", "Station B", "Station C", "Station C",
        "Station A", "Station A", "Station C", "Station D",
        "Station A", "Station B", "Station E",
        "Station A", "Station C", "Station E",
        "Station A", "Station D",
        "Station E", "Station E", "Station D"
    ],

    "started_at": pd.to_datetime([
        "2025-01-01 08:00", "2025-01-01 08:15", "2025-01-01 08:30",
        "2025-01-01 09:00", "2025-01-01 09:20",
        "2025-01-01 10:00", "2025-01-01 10:15", "2025-01-01 10:40",
        "2025-01-01 11:00",
        "2025-01-01 11:30", "2025-01-01 12:00", "2025-01-01 12:20",
        "2025-01-01 13:00", "2025-01-01 13:30", "2025-01-01 14:00",
        "2025-01-01 14:30", "2025-01-01 15:00",
        "2025-01-01 15:30", "2025-01-01 16:00", "2025-01-01 16:30"
    ]),

    "ended_at": pd.to_datetime([
        "2025-01-01 08:10", "2025-01-01 08:28", "2025-01-01 08:42",
        "2025-01-01 09:18", "2025-01-01 09:35",
        "2025-01-01 10:12", "2025-01-01 10:30", "2025-01-01 10:55",
        "2025-01-01 11:18",
        "2025-01-01 11:48", "2025-01-01 12:14", "2025-01-01 12:45",
        "2025-01-01 13:20", "2025-01-01 13:48", "2025-01-01 14:20",
        "2025-01-01 14:55", "2025-01-01 15:22",
        "2025-01-01 15:58", "2025-01-01 16:25", "2025-01-01 16:50"
    ]),

    "member_casual": [
        "member", "member", "casual", "member", "casual",
        "member", "member", "casual", "member",
        "casual", "member", "casual",
        "member", "casual", "member",
        "casual", "member",
        "member", "casual", "member"
    ]
})

trip_demo

,ride_id,start_station,end_station,started_at,ended_at,member_casual
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual
5,6,Station B,Station A,2025-01-01 10:00:00,2025-01-01 10:12:00,member
6,7,Station B,Station A,2025-01-01 10:15:00,2025-01-01 10:30:00,member
7,8,Station B,Station C,2025-01-01 10:40:00,2025-01-01 10:55:00,casual
8,9,Station B,Station D,2025-01-01 11:00:00,2025-01-01 11:18:00,member
9,10,Station C,Station A,2025-01-01 11:30:00,2025-01-01 11:48:00,casual


In [3]:
station_coordinates = pd.DataFrame({
    "station": [
        "Station A",
        "Station B",
        "Station C",
        "Station D",
        "Station E"
    ],
    "lat": [
        40.735,
        40.751,
        40.742,
        40.728,
        40.760
    ],
    "lng": [
        -73.991,
        -73.977,
        -73.985,
        -73.970,
        -73.995
    ]
})

station_coordinates

,station,lat,lng
0,Station A,40.735,-73.991
1,Station B,40.751,-73.977
2,Station C,40.742,-73.985
3,Station D,40.728,-73.970
4,Station E,40.760,-73.995


In [4]:
start_df = trip_demo.merge(station_coordinates, how='left', left_on='start_station', right_on='station')
start_df.rename(columns={
    'lat': 'start_lat',
    'lng': 'start_lng'
}, inplace=True)

In [5]:
start_df.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,station,start_lat,start_lng
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,Station A,40.735,-73.991
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,Station A,40.735,-73.991
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,Station A,40.735,-73.991
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,Station A,40.735,-73.991
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,Station A,40.735,-73.991


In [6]:
end_df = trip_demo.merge(station_coordinates, how='left', left_on='end_station', right_on='station')
end_df.rename(columns={
    'lat': 'end_lat',
    'lng': 'end_lng'
}, inplace=True)

end_df.drop(columns=['station'], inplace=True)

end_df.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,end_lat,end_lng
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,40.751,-73.977
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,40.751,-73.977
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,40.751,-73.977
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,40.742,-73.985
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,40.742,-73.985


In [7]:
start_df.columns
start_cols = ['ride_id', 'start_station', 'started_at', 'start_lat', 'start_lng']
end_cols = ['ride_id', 'end_station', 'ended_at', 'end_lat', 'end_lng', 'member_casual']

In [8]:
final_df = pd.merge(start_df[start_cols], end_df[end_cols], how='inner', on='ride_id')
final_df.head() 

,ride_id,start_station,started_at,start_lat,start_lng,end_station,ended_at,end_lat,end_lng,member_casual
0,1,Station A,2025-01-01 08:00:00,40.735,-73.991,Station B,2025-01-01 08:10:00,40.751,-73.977,member
1,2,Station A,2025-01-01 08:15:00,40.735,-73.991,Station B,2025-01-01 08:28:00,40.751,-73.977,member
2,3,Station A,2025-01-01 08:30:00,40.735,-73.991,Station B,2025-01-01 08:42:00,40.751,-73.977,casual
3,4,Station A,2025-01-01 09:00:00,40.735,-73.991,Station C,2025-01-01 09:18:00,40.742,-73.985,member
4,5,Station A,2025-01-01 09:20:00,40.735,-73.991,Station C,2025-01-01 09:35:00,40.742,-73.985,casual


### Map Center

In [9]:
map_center = [
    pd.concat([final_df['start_lat'], final_df['end_lat']]).mean(),
    pd.concat([final_df['start_lng'], final_df['end_lng']]).mean()
]

map_center

[np.float64(40.7427), np.float64(-73.9841)]

### Adding Duration

In [10]:
trip_demo["duration_min"] = (
    trip_demo['ended_at'] -trip_demo['started_at']
    ).dt.total_seconds() / 60 

trip_demo[[
    "ride_id",
    "start_station",
    "end_station",
    "started_at",
    "ended_at",
    "duration_min"
]].head()

,ride_id,start_station,end_station,started_at,ended_at,duration_min
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,10.0
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,13.0
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,12.0
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,18.0
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,15.0


### Adding Feature Time

In [11]:
trip_demo["date"] = trip_demo["started_at"].dt.date
trip_demo["hour"] = trip_demo["started_at"].dt.hour
trip_demo["day_name"] = trip_demo["started_at"].dt.day_name()
trip_demo["month_name"] = trip_demo["started_at"].dt.month_name()

trip_demo.head()

,ride_id,start_station,end_station,started_at,ended_at,member_casual,duration_min,date,hour,day_name,month_name
0,1,Station A,Station B,2025-01-01 08:00:00,2025-01-01 08:10:00,member,10.0,2025-01-01,8,Wednesday,January
1,2,Station A,Station B,2025-01-01 08:15:00,2025-01-01 08:28:00,member,13.0,2025-01-01,8,Wednesday,January
2,3,Station A,Station B,2025-01-01 08:30:00,2025-01-01 08:42:00,casual,12.0,2025-01-01,8,Wednesday,January
3,4,Station A,Station C,2025-01-01 09:00:00,2025-01-01 09:18:00,member,18.0,2025-01-01,9,Wednesday,January
4,5,Station A,Station C,2025-01-01 09:20:00,2025-01-01 09:35:00,casual,15.0,2025-01-01,9,Wednesday,January


### Point

In [12]:
map_center

[np.float64(40.7427), np.float64(-73.9841)]

In [13]:
m = folium.Map(
    location=map_center,
    zoom_start=13
)

for _, row in station_coordinates.iterrows():
    folium.Marker(
        location=[row['lat'], row['lng']],
        popup=row['station']
    ).add_to(m)
    
m

### Line 

In [21]:
start_point = [final_df.loc[0, 'start_lat'], final_df.loc[0, 'start_lng']]
end_point = [final_df.loc[0, 'end_lat'], final_df.loc[0, 'end_lng']]

folium.Marker(start_point, popup='Start').add_to(m)
folium.Marker(end_point, popup='End').add_to(m)

folium.PolyLine(
    locations=[start_point, end_point],
    weight=5,
    opacity=0.8
).add_to(m)

m